# GPTQ From Scratch — Full Benchmark

Ce notebook lance **toutes** les experiences de quantization sur GPT-2 (124M) :
- Baseline FP32/FP16
- GPTQ 4-bit, 3-bit, 2-bit
- Naive RTN 4-bit, 3-bit, 2-bit

## Instructions
1. Va dans **Runtime > Change runtime type > GPU (T4)**
2. Clique **Runtime > Run all**
3. Attends ~20 min
4. Le tableau final s'affiche a la fin

## 1. Setup

In [ ]:
!pip install -q torch transformers datasets tqdm numpy

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU")

## 2. Source files
On ecrit tous les fichiers source directement dans Colab avec `%%writefile`.

In [ ]:
%%writefile quantize.py
"""
Quantization and dequantization utilities.

Symmetric uniform quantization to n-bit integers.
"""

import torch


def quantize_tensor(w, n_bits=4):
    qmax = 2 ** (n_bits - 1) - 1
    qmin = -(2 ** (n_bits - 1))
    scale = w.abs().max() / qmax
    scale = scale.clamp(min=1e-10)
    q = torch.clamp(torch.round(w / scale), qmin, qmax)
    w_hat = q * scale
    return w_hat, scale


def compute_row_scales(W, n_bits=4):
    qmax = 2 ** (n_bits - 1) - 1
    scales = W.abs().amax(dim=1) / qmax
    scales = scales.clamp(min=1e-10)
    return scales


def quantize_column(w_col, scales, n_bits=4):
    qmax = 2 ** (n_bits - 1) - 1
    qmin = -(2 ** (n_bits - 1))
    q = torch.clamp(torch.round(w_col / scales), qmin, qmax)
    return q * scales


def round_to_nearest(w, n_bits=4):
    w_hat, _ = quantize_tensor(w, n_bits)
    return w_hat

In [ ]:
%%writefile model_utils.py
"""
Model utilities: GPT-2 loading, calibration data preparation, block-wise processing.
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset


def load_model(model_name="gpt2", device="cuda"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype)
    model = model.to(device)
    model.eval()
    return model, tokenizer


def get_calibration_data(tokenizer, n_samples=128, seq_len=2048, seed=42):
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    text = "\n\n".join(dataset["text"])
    tokens = tokenizer.encode(text)
    tokens = torch.tensor(tokens, dtype=torch.long)
    rng = torch.Generator()
    rng.manual_seed(seed)
    samples = []
    for _ in range(n_samples):
        start = torch.randint(0, len(tokens) - seq_len, (1,), generator=rng).item()
        segment = tokens[start : start + seq_len].unsqueeze(0)
        samples.append(segment)
    return samples


def get_transformer_blocks(model):
    transformer = model.transformer
    return (
        transformer.h,
        transformer.wte,
        transformer.wpe,
        transformer.ln_f,
    )


@torch.no_grad()
def get_block_inputs(model, calibration_data, device="cpu"):
    blocks, wte, wpe, ln_f = get_transformer_blocks(model)
    block_inputs = []
    for batch in calibration_data:
        batch = batch.to(device)
        position_ids = torch.arange(0, batch.shape[1], device=device).unsqueeze(0)
        hidden = wte(batch) + wpe(position_ids)
        block_inputs.append(hidden)
    return block_inputs


def get_weight_and_type(layer):
    if type(layer).__name__ == "Conv1D":
        return layer.weight.data.T.clone(), True
    else:
        return layer.weight.data.clone(), False


def set_weight(layer, Q, is_conv1d):
    if is_conv1d:
        layer.weight.data = Q.T.to(layer.weight.dtype)
    else:
        layer.weight.data = Q.to(layer.weight.dtype)

In [ ]:
%%writefile gptq.py
"""
GPTQ algorithm implementation (Frantar et al., 2022).
"""

import time
import torch
from tqdm import tqdm
from quantize import compute_row_scales, quantize_column


def compute_hessian(X, damp_pct=0.01):
    n = X.shape[0]
    H = X.T @ X / n
    damp = damp_pct * torch.mean(torch.diag(H))
    diag_idx = range(H.shape[0])
    H[diag_idx, diag_idx] += damp
    return H


def gptq_quantize_layer(W, H, n_bits=4, block_size=128):
    W = W.clone().float()
    n_rows, n_cols = W.shape
    try:
        L = torch.linalg.cholesky(H)
        H_inv = torch.cholesky_inverse(L)
    except RuntimeError:
        extra_damp = 0.1 * torch.mean(torch.diag(H))
        H_reg = H.clone()
        diag_idx = range(H.shape[0])
        H_reg[diag_idx, diag_idx] += extra_damp
        L = torch.linalg.cholesky(H_reg)
        H_inv = torch.cholesky_inverse(L)
    scales = compute_row_scales(W, n_bits)
    Q = torch.zeros_like(W)
    loss = 0.0
    for i1 in range(0, n_cols, block_size):
        i2 = min(i1 + block_size, n_cols)
        count = i2 - i1
        W_blk = W[:, i1:i2].clone()
        Q_blk = torch.zeros_like(W_blk)
        Err = torch.zeros_like(W_blk)
        H_inv_blk = H_inv[i1:i2, i1:i2]
        for j in range(count):
            w_col = W_blk[:, j]
            h_jj = H_inv_blk[j, j]
            q_col = quantize_column(w_col, scales, n_bits)
            Q_blk[:, j] = q_col
            err = (w_col - q_col) / h_jj
            Err[:, j] = err
            loss += ((w_col - q_col) ** 2 / h_jj).sum().item()
            if j < count - 1:
                W_blk[:, j + 1:] -= err.unsqueeze(1) * H_inv_blk[j, j + 1:].unsqueeze(0)
        Q[:, i1:i2] = Q_blk
        if i2 < n_cols:
            W[:, i2:] -= Err @ H_inv[i1:i2, i2:]
    return Q, loss


@torch.no_grad()
def quantize_model(model, calibration_data, n_bits=4, block_size=128, device="cpu"):
    from model_utils import get_transformer_blocks, get_block_inputs, get_weight_and_type, set_weight
    blocks, wte, wpe, ln_f = get_transformer_blocks(model)
    print("Computing embedding outputs for calibration data...")
    hidden_states = get_block_inputs(model, calibration_data, device=device)
    print(f"Got {len(hidden_states)} calibration samples")
    stats = {}
    for block_idx, block in enumerate(tqdm(blocks, desc=f"GPTQ {n_bits}-bit blocks")):
        layers_to_quantize = []
        for name, module in block.named_modules():
            if isinstance(module, torch.nn.Linear) or type(module).__name__ == "Conv1D":
                layers_to_quantize.append((name, module))
        layer_inputs = {name: [] for name, _ in layers_to_quantize}
        hooks = []
        for name, module in layers_to_quantize:
            def make_hook(layer_name):
                def hook_fn(mod, inp, out):
                    layer_inputs[layer_name].append(
                        inp[0].detach().reshape(-1, inp[0].shape[-1])
                    )
                return hook_fn
            hooks.append(module.register_forward_hook(make_hook(name)))
        for h in hidden_states:
            block(h)
        for handle in hooks:
            handle.remove()
        for name, module in layers_to_quantize:
            t0 = time.time()
            full_name = f"transformer.h.{block_idx}.{name}"
            X = torch.cat(layer_inputs[name], dim=0)
            W, is_conv1d = get_weight_and_type(module)
            H = compute_hessian(X.float())
            Q, loss = gptq_quantize_layer(W.float(), H, n_bits=n_bits, block_size=block_size)
            set_weight(module, Q, is_conv1d)
            elapsed = time.time() - t0
            stats[full_name] = {"loss": loss, "time": elapsed}
            del X, H, W, Q
        new_hidden = []
        for h in hidden_states:
            new_hidden.append(block(h)[0].detach())
        hidden_states = new_hidden
    return stats

In [ ]:
%%writefile evaluate.py
"""
Perplexity evaluation on WikiText-2 test set.
"""

import torch
from datasets import load_dataset
from tqdm import tqdm


@torch.no_grad()
def evaluate_perplexity(model, tokenizer, device="cuda", stride=512):
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n\n".join(dataset["text"])
    encodings = tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids.to(device)
    max_len = model.config.n_positions
    total_len = input_ids.size(1)
    nlls = []
    n_tokens = 0
    for begin in tqdm(range(0, total_len, stride), desc="Evaluating perplexity"):
        end = min(begin + max_len, total_len)
        input_chunk = input_ids[:, begin:end]
        labels = input_chunk.clone()
        n_ctx = 0 if begin == 0 else max_len - stride
        labels[:, :n_ctx] = -100
        outputs = model(input_chunk, labels=labels)
        n_valid = (labels != -100).sum().item()
        if n_valid > 0:
            nlls.append(outputs.loss.float() * n_valid)
            n_tokens += n_valid
        if end == total_len:
            break
    avg_nll = torch.stack(nlls).sum() / n_tokens
    perplexity = torch.exp(avg_nll).item()
    return perplexity

## 3. Run all experiments

On lance tout automatiquement. Chaque run recharge le modele frais.

In [ ]:
import time
import torch
from model_utils import load_model, get_calibration_data
from evaluate import evaluate_perplexity
from gptq import quantize_model
from quantize import round_to_nearest

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STRIDE = 512
N_SAMPLES = 128
BLOCK_SIZE = 128

results = {}

def run_experiment(name, bits=None, method=None):
    """Run one experiment: load fresh model, quantize, evaluate."""
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    model, tokenizer = load_model("gpt2", device=DEVICE)
    seq_len = min(model.config.n_positions, 2048)

    quant_time = 0.0

    if method == "gptq":
        calib_data = get_calibration_data(tokenizer, n_samples=N_SAMPLES, seq_len=seq_len)
        print(f"Calibration: {len(calib_data)} samples x {seq_len} tokens")
        t0 = time.time()
        stats = quantize_model(model, calib_data, n_bits=bits, block_size=BLOCK_SIZE, device=DEVICE)
        quant_time = time.time() - t0
        total_loss = sum(s["loss"] for s in stats.values())
        print(f"GPTQ done in {quant_time:.1f}s, loss={total_loss:.2f}")
        del calib_data

    elif method == "rtn":
        t0 = time.time()
        for mod_name, module in model.named_modules():
            if isinstance(module, torch.nn.Linear):
                module.weight.data = round_to_nearest(module.weight.data, bits)
            elif type(module).__name__ == "Conv1D":
                module.weight.data = round_to_nearest(module.weight.data, bits)
        quant_time = time.time() - t0
        print(f"RTN done in {quant_time:.1f}s")

    t0 = time.time()
    ppl = evaluate_perplexity(model, tokenizer, device=DEVICE, stride=STRIDE)
    eval_time = time.time() - t0
    print(f"Perplexity: {ppl:.2f} (eval: {eval_time:.1f}s)")

    results[name] = {
        "perplexity": ppl,
        "quant_time": quant_time,
        "eval_time": eval_time,
    }

    # Free memory
    del model, tokenizer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return ppl

In [ ]:
# 1. Baseline
run_experiment("FP16 baseline", method=None)

In [ ]:
# 2. GPTQ 4-bit
run_experiment("GPTQ 4-bit", bits=4, method="gptq")

In [ ]:
# 3. GPTQ 3-bit
run_experiment("GPTQ 3-bit", bits=3, method="gptq")

In [ ]:
# 4. GPTQ 2-bit
run_experiment("GPTQ 2-bit", bits=2, method="gptq")

In [ ]:
# 5. Naive RTN 4-bit
run_experiment("RTN 4-bit", bits=4, method="rtn")

In [ ]:
# 6. Naive RTN 3-bit
run_experiment("RTN 3-bit", bits=3, method="rtn")

In [ ]:
# 7. Naive RTN 2-bit
run_experiment("RTN 2-bit", bits=2, method="rtn")

## 4. Results table

In [ ]:
# Pretty print results
baseline_ppl = results.get("FP16 baseline", {}).get("perplexity", "N/A")

print("\n" + "=" * 65)
print("  FINAL RESULTS — WikiText-2 Perplexity on GPT-2 (124M)")
print("=" * 65)
print(f"\nBaseline (FP16): {baseline_ppl}")
print()
print(f"{'Method':<20} {'4-bit':>10} {'3-bit':>10} {'2-bit':>10}")
print("-" * 52)

for method in ["GPTQ", "RTN"]:
    vals = []
    for bits in [4, 3, 2]:
        key = f"{method} {bits}-bit"
        if key in results:
            vals.append(f"{results[key]['perplexity']:.2f}")
        else:
            vals.append("N/A")
    print(f"{method:<20} {vals[0]:>10} {vals[1]:>10} {vals[2]:>10}")

print("-" * 52)
print()

# Markdown format for easy copy-paste
print("\nMARKDOWN TABLE (copy this):")
print("```")
print("| Method | 4-bit | 3-bit | 2-bit |")
print("|:--|--:|--:|--:|")
print(f"| FP16 baseline | {baseline_ppl} | {baseline_ppl} | {baseline_ppl} |")
for method in ["GPTQ", "RTN"]:
    vals = []
    for bits in [4, 3, 2]:
        key = f"{method} {bits}-bit"
        if key in results:
            vals.append(f"{results[key]['perplexity']:.2f}")
        else:
            vals.append("TBD")
    prefix = "**GPTQ**" if method == "GPTQ" else "Naive RTN"
    if method == "GPTQ":
        print(f"| {prefix} | **{vals[0]}** | **{vals[1]}** | **{vals[2]}** |")
    else:
        print(f"| {prefix} | {vals[0]} | {vals[1]} | {vals[2]} |")
print("```")

print("\n\nTIMINGS:")
for name, r in results.items():
    print(f"  {name}: quant={r['quant_time']:.1f}s, eval={r['eval_time']:.1f}s")

In [ ]:
# Save results to JSON (optional backup)
import json
with open("results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Results saved to results.json")
print("\nDone!")